## Data Ingestion using `Auto Loader`

### Clean the Entire Setup from the previous Streaming Lab

In [0]:
%sql

DROP TABLE IF EXISTS ev_lab.bronze.ev_charging_sessions_stream

In [0]:
dbutils.fs.rm(
    "/Volumes/ev_lab/bronze/raw_files/ev_sessions_stream/",
    recurse=True
)

dbutils.fs.rm(
    "/Volumes/ev_lab/bronze/raw_files/checkpoints/",
    recurse=True
)

dbutils.fs.rm(
    "/Volumes/ev_lab/bronze/raw_files/schema/ev_sessions/",
    recurse=True
)

### Setup Paths Again (Volume-Based)

In [0]:
input_path = "/Volumes/ev_lab/bronze/raw_files/ev_sessions_stream/"
checkpoint_path = "/Volumes/ev_lab/bronze/raw_files/checkpoints/ev_sessions/"
schema_path = "/Volumes/ev_lab/bronze/raw_files/schema/ev_sessions/"

dbutils.fs.mkdirs(input_path)
dbutils.fs.mkdirs(checkpoint_path)
dbutils.fs.mkdirs(schema_path)

### Read Input Path as Streaming Source

In [0]:
stream_df = spark.readStream \
        .format("cloudFiles") \
        .option("cloudFiles.format", "csv") \
        .option("cloudFiles.inferColumnTypes", "true") \
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
        .option("cloudFiles.schemaLocation", schema_path) \
        .load(input_path)

### Simulate Incoming Data (New File Arrives!)

In [0]:
data1 = """session_id,station_id,charger_id,start_time,end_time,energy_kwh,charging_rate_kw,temperature_c,status
S101,EV_IND_001,C-IND-001,2025-06-01 10:00:00,2025-06-01 11:00:00,40,40,35,Completed
S102,EV_IND_001,C-IND-002,2025-06-01 10:15:00,2025-06-01 11:10:00,38,35,34,Completed
"""

with open(f"{input_path}/batch1.csv", "w") as f:
    f.write(data1)

### Write to a Bronze Delta Table

In [0]:
from pyspark.sql.utils import StreamingQueryException

# Handle schema evolution with triggered micro-batches
for attempt in range(3):
    stream_df = spark.readStream \
        .format("cloudFiles") \
        .option("cloudFiles.format", "csv") \
        .option("cloudFiles.inferColumnTypes", "true") \
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
        .option("cloudFiles.schemaLocation", schema_path) \
        .load(input_path)

    q = stream_df.writeStream \
        .format("delta") \
        .outputMode("append") \
        .option("checkpointLocation", checkpoint_path) \
        .option("mergeSchema", "true") \
        .trigger(availableNow=True) \
        .toTable("ev_lab.bronze.ev_charging_sessions_stream")
    try:
        q.awaitTermination()
        break
    except StreamingQueryException as e:
        if "UNKNOWN_FIELD_EXCEPTION" in str(e):
            print(f"Schema evolution detected, restarting... (attempt {attempt + 1})")
            continue
        raise

# Start continuous stream with the evolved schema
stream_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .option("cloudFiles.schemaLocation", schema_path) \
    .load(input_path)

query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .toTable("ev_lab.bronze.ev_charging_sessions_stream")

In [0]:
%sql

SELECT * FROM ev_lab.bronze.ev_charging_sessions_stream

### Simulate more Incoming Data (Next File Batch)

In [0]:
data2 = """session_id,station_id,charger_id,start_time,end_time,energy_kwh,charging_rate_kw,temperature_c,status
S103,EV_IND_001,C-IND-003,2025-06-01 11:30:00,2025-06-01 12:00:00,0,0,33,Fault
S104,EV_IND_001,C-IND-004,2025-06-01 12:00:00,2025-06-01 13:00:00,45,45,36,Completed
"""

with open(f"{input_path}/batch2.csv", "w") as f:
    f.write(data2)

In [0]:
%sql

SELECT * FROM ev_lab.bronze.ev_charging_sessions_stream

### See Schema Evolution in Action

In [0]:
data3 = """session_id,station_id,charger_id,start_time,end_time,energy_kwh,charging_rate_kw,temperature_c,status,voltage,current,charger_type
S105,EV_IND_001,C-IND-003,2025-06-01 11:30:00,2025-06-01 12:00:00,0,0,33,Fault,400,0,Fast
S106,EV_IND_001,C-IND-004,2025-06-01 12:00:00,2025-06-01 13:00:00,45,45,36,Completed,415,108,Fast
"""

with open(f"{input_path}/batch3.csv", "w") as f:
    f.write(data3)

In [0]:
%sql

SELECT * FROM ev_lab.bronze.ev_charging_sessions_stream

### Stop the Stream

In [0]:
query.stop()